### Import libraries

In [1]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### Add src to path

In [2]:
sys.path.append(os.path.abspath('..'))

### Import out modules

In [3]:
from src.model import train_model, evaluate_model, tune_hyperparameters, save_model, load_model
from src.predict import create_submission

### Load processed data

In [4]:
train = pd.read_csv('../data/processed/train_processed.csv')
val = pd.read_csv('../data/processed/val_processed.csv')
test = pd.read_csv('../data/processed/test_processed.csv')

print(f"Train shape: {train.shape}")
print(f"Val shape: {val.shape}")
print(f"Test shape: {test.shape}")

Train shape: (712, 21)
Val shape: (179, 21)
Test shape: (418, 21)


### Prepare X and y

In [5]:
X_train = train.drop("Survived", axis =1)
y_train = train["Survived"]

X_val = val.drop('Survived', axis = 1)
y_val = val['Survived']

### For test, we need the PassengerId for submission (test still has PassengerId)
#### We'll keep it separate.

In [6]:
passenger_ids = test['PassengerId']
X_test = test.drop('PassengerId', axis=1)

print("\nFeatures used for training: ", X_train.columns.tolist())


Features used for training:  ['Pclass', 'Sex', 'Fare', 'AgeGroupOrdinal', 'IsAlone', 'Title_Col', 'Title_Countess', 'Title_Dr', 'Title_Lady', 'Title_Major', 'Title_Master', 'Title_Miss', 'Title_Mlle', 'Title_Mme', 'Title_Mr', 'Title_Mrs', 'Title_Ms', 'Title_Rev', 'Embarked_Q', 'Embarked_S']


### Train Baseline Models

In [7]:
# Logistic regerssion
print("\n" + "="*50)
print("Training Logistic Regression...")

log_model = train_model(X_train, y_train, model_type='logistic')
y_pred_log, acc_log = evaluate_model(log_model, X_val, y_val)


Training Logistic Regression...
Validation Accuracy: 0.7821

Confusion Matrix:
               precision    recall  f1-score   support

           0       0.82      0.81      0.81       105
           1       0.73      0.74      0.74        74

    accuracy                           0.78       179
   macro avg       0.78      0.78      0.78       179
weighted avg       0.78      0.78      0.78       179


Confusion Matrix:
 [[85 20]
 [19 55]]


In [8]:
# Random Forest
print("\n" + "="*50)
print("Training Random Forest...")
rf_model = train_model(X_train, y_train, model_type='random_forest', n_estimators=100, max_depth=10)
y_pred_rf , acc_rf = evaluate_model(rf_model, X_val, y_val)


Training Random Forest...
Validation Accuracy: 0.8268

Confusion Matrix:
               precision    recall  f1-score   support

           0       0.84      0.87      0.85       105
           1       0.80      0.77      0.79        74

    accuracy                           0.83       179
   macro avg       0.82      0.82      0.82       179
weighted avg       0.83      0.83      0.83       179


Confusion Matrix:
 [[91 14]
 [17 57]]


In [9]:
# XGBoost
print("\n" + "="*50)
print("Training XGBoost...")
xgb_model = train_model(X_train, y_train, model_type='xgboost', n_estimators=100, max_depth=6, learning_rate=0.1)
y_pred_xgb, acc_xgb = evaluate_model(xgb_model, X_val, y_val)


Training XGBoost...
Validation Accuracy: 0.8380

Confusion Matrix:
               precision    recall  f1-score   support

           0       0.86      0.87      0.86       105
           1       0.81      0.80      0.80        74

    accuracy                           0.84       179
   macro avg       0.83      0.83      0.83       179
weighted avg       0.84      0.84      0.84       179


Confusion Matrix:
 [[91 14]
 [15 59]]


d:\Programming\Python\titanic-project\venv\lib\site-packages\xgboost\training.py:200: UserWarning: [17:32:06] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


### Compare Performances

In [10]:
print("\n" + "="*50)
print("Summary of Validation Accuracies: ")
print(f"Logistic Regression: {acc_log:.4f}")
print(f"Random Forest:       {acc_rf:.4f}")
print(f"XGBoost:             {acc_xgb:.4f}")


Summary of Validation Accuracies: 
Logistic Regression: 0.7821
Random Forest:       0.8268
XGBoost:             0.8380


### Hyperparameter Tuning

In [11]:
# Choose the best performing model (e.g., XGBoost) and tune it.
print("\n" + "="*50)
print("Hyperparameter Tuning for XGBoost...")
best_xgb, best_params = tune_hyperparameters(X_train, y_train, model_type='xgboost')

# Evaluate tuned model
y_pred_tuned, acc_tuned = evaluate_model(best_xgb, X_val, y_val)
print(f"Tuned XGBoost Validation Accuracy: {acc_tuned:.4f}")


Hyperparameter Tuning for XGBoost...
Best parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}
Best cross_validation accuracy: 0.8300
Validation Accuracy: 0.8156

Confusion Matrix:
               precision    recall  f1-score   support

           0       0.83      0.86      0.85       105
           1       0.79      0.76      0.77        74

    accuracy                           0.82       179
   macro avg       0.81      0.81      0.81       179
weighted avg       0.81      0.82      0.82       179


Confusion Matrix:
 [[90 15]
 [18 56]]
Tuned XGBoost Validation Accuracy: 0.8156


d:\Programming\Python\titanic-project\venv\lib\site-packages\xgboost\training.py:200: UserWarning: [17:37:26] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


### Save the Best Model

In [12]:
# Choose the best model (here we use the tuned XGBoost)
best_model = best_xgb
save_model(best_model, '../models/best_model.pkl')

Model saved to ../models/best_model.pkl


### Generate Submission File

In [13]:
print("\n" + "="*50)
print("Generating Submission File...")
submission = create_submission(best_model, X_test, passenger_ids, filename="submission_xgboost_tuned.csv")


Generating Submission File...
Submission file saved to ../submissions\submission_xgboost_tuned.csv


In [14]:
print(submission.head())

   PassengerId  Survived
0          892         0
1          893         1
2          894         0
3          895         0
4          896         1
